## Data Wrangling to create a treemap in Looker Studio

Import the necessary libraries

In [1]:
# Data wrangling and numeric operations
import pandas as pd
import numpy as np

# Date manipulation
import datetime as dt

Import the data

In [2]:
new_db = pd.read_csv('2market_export.csv')

Inspect the df statistics

In [3]:
new_db.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2213 entries, 0 to 2212
Data columns (total 28 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              2213 non-null   int64  
 1   year_birth      2213 non-null   int64  
 2   education       2213 non-null   object 
 3   marital_status  2209 non-null   object 
 4   income          2213 non-null   float64
 5   kidhome         2213 non-null   int64  
 6   teenhome        2213 non-null   int64  
 7   dt_customer     2213 non-null   object 
 8   recency         2213 non-null   int64  
 9   amtliq          2213 non-null   int64  
 10  amtvege         2213 non-null   int64  
 11  amtnonveg       2213 non-null   int64  
 12  amtpes          2213 non-null   int64  
 13  amtchocolates   2213 non-null   int64  
 14  amtcomm         2213 non-null   int64  
 15  numdeals        2213 non-null   int64  
 16  numwebbuy       2213 non-null   int64  
 17  numwalkinpur    2213 non-null   i

In [4]:
new_db.head()

,id,year_birth,education,marital_status,income,kidhome,teenhome,dt_customer,recency,amtliq,...,numvisits,response,complain,country,count_success,bulkmail_ad,twitter_ad,instagram_ad,facebook_ad,brochure_ad
0,6663,1941,PhD,Single,51141.0,0,0,2013-07-08,96,144,...,5,0,0,SP,0,0,0,0,0,0
1,6932,1942,PhD,Married,93027.0,0,0,2013-04-13,77,1285,...,2,0,0,SP,1,0,0,1,0,0
2,1453,1944,PhD,Widow,57513.0,0,0,2013-07-06,59,735,...,6,0,0,SP,0,0,0,0,0,0
3,2968,1944,PhD,Divorced,48948.0,0,0,2013-02-01,53,437,...,6,1,0,AUS,1,1,0,0,0,0
4,4994,1944,Master,Single,77598.0,0,0,2013-10-01,53,1193,...,3,0,0,SP,1,0,0,1,0,0


Noticed dt_customer did not have the correct data type. Changed that and checked changes.

In [5]:
# change dtype of dt_customer using datetime
new_db['dt_customer'] = pd.to_datetime(new_db['dt_customer'], errors='coerce')

In [6]:
new_db.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2213 entries, 0 to 2212
Data columns (total 28 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   id              2213 non-null   int64         
 1   year_birth      2213 non-null   int64         
 2   education       2213 non-null   object        
 3   marital_status  2209 non-null   object        
 4   income          2213 non-null   float64       
 5   kidhome         2213 non-null   int64         
 6   teenhome        2213 non-null   int64         
 7   dt_customer     2213 non-null   datetime64[ns]
 8   recency         2213 non-null   int64         
 9   amtliq          2213 non-null   int64         
 10  amtvege         2213 non-null   int64         
 11  amtnonveg       2213 non-null   int64         
 12  amtpes          2213 non-null   int64         
 13  amtchocolates   2213 non-null   int64         
 14  amtcomm         2213 non-null   int64         
 15  numd

Transforms the marketing dataset from wide to long format to prepare for treemap visualization. It melts six separate product revenue columns into two key columns: product_type (the dimension) and revenue (the metric), while preserving all other customer attributes. The product codes are also renamed with human-readable business category labels before the final dataset is exported as a CSV file.

In [7]:
# Melt the 6 product columns, keeping all other columns as identifiers
new_db_melted = pd.melt(
    new_db,
    id_vars=[
        'id', 'year_birth', 'education', 'marital_status', 'income',
        'kidhome', 'teenhome', 'dt_customer', 'recency', 'numdeals',
        'numwebbuy', 'numwalkinpur', 'numvisits', 'response', 'complain',
        'country', 'count_success', 'bulkmail_ad', 'twitter_ad',
        'instagram_ad', 'facebook_ad', 'brochure_ad'
    ],
    value_vars=[
        'amtliq', 'amtvege', 'amtnonveg', 'amtpes', 
        'amtchocolates', 'amtcomm'
    ],
    var_name='product_type',  # This becomes your dimension for the treemap
    value_name='revenue'      # This becomes your metric for the treemap
)

# Rename product_type values to your cosmetic labels
label_mapping = {
    'amtliq': 'Beer, Wine & Spirits',
    'amtvege': 'Greengrocer',
    'amtnonveg': 'Butcher',
    'amtpes': 'Fishmonger',
    'amtchocolates': 'Confectionary',
    'amtcomm': 'Commodities'
}
new_db_melted['product_type'] = new_db_melted['product_type'].map(label_mapping)

# Export to CSV
new_db_melted.to_csv('marketing_data_melted.csv', index=False)

print("Melted DataFrame shape:", new_db_melted.shape)
print("Original DataFrame shape:", new_db.shape)
print(f"File saved: marketing_data_melted.csv")

Melted DataFrame shape: (13278, 24)
Original DataFrame shape: (2213, 28)
File saved: marketing_data_melted.csv


In [8]:
new_db_melted.head()

,id,year_birth,education,marital_status,income,kidhome,teenhome,dt_customer,recency,numdeals,...,complain,country,count_success,bulkmail_ad,twitter_ad,instagram_ad,facebook_ad,brochure_ad,product_type,revenue
0,6663,1941,PhD,Single,51141.0,0,0,2013-07-08,96,1,...,0,SP,0,0,0,0,0,0,"Beer, Wine & Spirits",144
1,6932,1942,PhD,Married,93027.0,0,0,2013-04-13,77,0,...,0,SP,1,0,0,1,0,0,"Beer, Wine & Spirits",1285
2,1453,1944,PhD,Widow,57513.0,0,0,2013-07-06,59,2,...,0,SP,0,0,0,0,0,0,"Beer, Wine & Spirits",735
3,2968,1944,PhD,Divorced,48948.0,0,0,2013-02-01,53,2,...,0,AUS,1,1,0,0,0,0,"Beer, Wine & Spirits",437
4,4994,1944,Master,Single,77598.0,0,0,2013-10-01,53,1,...,0,SP,1,0,0,1,0,0,"Beer, Wine & Spirits",1193
